# 01 — Audit de Stage 3 et recherche de ses limites

**But :** comprendre le champion privé à **92,5**, vérifier ce qu'il sait réellement faire sans LLM, puis déterminer où il faut chercher les prochains gains.

Ce notebook sépare strictement quatre statuts :

- **MEASURED** : résultat réellement observé dans nos tests ou sur ML-Arena ;
- **INFERRED** : déduit d'un score affiché et de tailles de familles connues ;
- **HYPOTHESIS** : explication plausible qui reste à tester ;
- **UNKNOWN** : information inaccessible avec les données publiques.

> Ce notebook ne cherche pas à prouver que Stage 3 est mauvais. Il cherche à savoir **où il est déjà solide et où son comportement reste inconnu**.

In [1]:
from pathlib import Path
import sys, json, pandas as pd, numpy as np

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p/'README.md').exists() and (p/'src/evaluation').exists() and (p/'agents').exists():
            return p
    raise FileNotFoundError('Lancez le notebook depuis le dépôt DS_Harness_Experiments.')

ROOT=find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))

from src.evaluation.core import load_tasks, load_agent_module, evaluate_deterministic
from src.evaluation.metrics import summarize_audit, family_summary
print('ROOT détecté =', ROOT.name)

ROOT détecté = projects


## 1. Le champion privé réellement observé

In [2]:
private=json.loads((ROOT/'results/reference/private_champion.json').read_text(encoding='utf-8'))
pd.DataFrame([{
    'statut':private['status'],
    'score privé':private['score'],
    'No-files':private['category_scores']['nofiles'],
    'Tables':private['category_scores']['tables'],
    'Fits':private['category_scores']['fits'],
    'answered':private['answered'],
    'format_errors':private['format_errors'],
}])

,statut,score privé,No-files,Tables,Fits,answered,format_errors
0,MEASURED_ON_PRIVATE_PLATFORM,92.5,95.6,88.0,96.0,120,0


**MEASURED :** le seul score privé validé reste **92,5/100**. Le score total correspond à 111 bonnes réponses sur 120.

**INFERRED :** avec les tailles privées connues (45 / 50 / 25), les sous-scores affichés sont compatibles avec 43/45 No-files, 44/50 Tables et 24/25 Fits. Cette reconstruction est très forte, mais ce ne sont pas des identifiants d'erreurs privés : nous ne savons pas quelles tâches exactes ont échoué.

## 2. Composition du benchmark public

In [3]:
dev=load_tasks(ROOT/'data/public/dev.json')
composition=(pd.DataFrame([{'id':t['id'],'type':t['type'],'family':t['type'].split('.')[0]} for t in dev])
             .groupby(['family','type']).size().rename('n').reset_index())
print('Total:',len(dev))
display(composition)
display(composition.groupby('family',as_index=False)['n'].sum())

Total: 180


,family,type,n
0,calc,calc.arithmetic,8
1,calc,calc.dates,8
2,calc,calc.functions,7
3,calc,calc.growth,7
4,calc,calc.integer,7
5,calc,calc.percent,7
6,calc,calc.probability,7
7,calc,calc.stats_inline,7
8,calc,calc.word,7
9,fit,fit.linear,18


,family,n
0,calc,65
1,fit,35
2,table,80


## 3. Audit déterministe frais de Stage 3

In [4]:
stage3=load_agent_module(ROOT/'agents/champions/stage3_92_5.py','nb1_stage3')
audit=evaluate_deterministic(stage3,dev,'dev')
display(family_summary(audit))
print('TOTAL =',summarize_audit(audit))

,family,n,covered,correct,wrong,coverage,precision_if_covered,selective_risk,deterministic_accuracy
0,calc,65,25,25,0,0.384615,1.0,0.0,0.384615
1,fit,35,35,35,0,1.000000,1.0,0.0,1.000000
2,table,80,80,80,0,1.000000,1.0,0.0,1.000000


TOTAL = {'n': 180, 'covered': 140, 'correct': 140, 'wrong': 0, 'coverage': 0.7777777777777778, 'precision_if_covered': 1.0, 'selective_risk': 0.0, 'deterministic_accuracy': 0.7777777777777778}


Résultat attendu :

- No-files : **25/65** couverts, **25/25** corrects parmi les couverts ;
- Tables : **80/80**, **80/80** ;
- Fits : **35/35**, **35/35** ;
- total : **140/180** tâches interceptées déterministiquement et **140/140** correctes.

La métrique essentielle ici n'est pas seulement la couverture : c'est la **précision conditionnelle** des règles.

$$
P_{det}=\frac{\#\text{réponses déterministes correctes}}{\#\text{réponses déterministes}}
$$

Sur le dev, Stage 3 a donc \(P_{det}=1\) sur 140 décisions déterministes.

## 4. Où sont les 40 tâches laissées au LLM ?

In [5]:
by_type=(audit.groupby('type')
         .agg(n=('id','size'),covered=('covered','sum'),correct=('correct','sum'),wrong=('wrong','sum'))
         .reset_index())
by_type['coverage_pct']=100*by_type.covered/by_type.n
display(by_type)
print('Tâches dev non interceptées par le déterministe:', int((~audit.covered).sum()))
display(audit.loc[~audit.covered,['id','type']].groupby('type').size().rename('n').reset_index())

,type,n,covered,correct,wrong,coverage_pct
0,calc.arithmetic,8,1,1,0,12.500000
1,calc.dates,8,0,0,0,0.000000
2,calc.functions,7,0,0,0,0.000000
3,calc.growth,7,5,5,0,71.428571
4,calc.integer,7,5,5,0,71.428571
5,calc.percent,7,3,3,0,42.857143
6,calc.probability,7,5,5,0,71.428571
7,calc.stats_inline,7,0,0,0,0.000000
8,calc.word,7,6,6,0,85.714286
9,fit.linear,18,18,18,0,100.000000


Tâches dev non interceptées par le déterministe: 40


,type,n
0,calc.arithmetic,7
1,calc.dates,8
2,calc.functions,7
3,calc.growth,2
4,calc.integer,2
5,calc.percent,4
6,calc.probability,2
7,calc.stats_inline,7
8,calc.word,1


**MEASURED :** toutes les Tables et tous les Fits publics sont déjà pris en charge par les règles. Les 40 abstentions publiques de Stage 3 sont des No-files.

Cela donne deux conclusions différentes :

1. **No-files** : il existe une opportunité de retirer au LLM des calculs simples, mais seulement si l'extension reste sélective.
2. **Tables/Fits** : améliorer encore les mêmes templates publics a un faible rendement scientifique. Le privé étant inférieur au public sur ces familles, il faut chercher des **variantes, compositions, cas OOD et edge cases**, sans prétendre connaître les erreurs privées.

## 5. Stress tests métamorphiques de Stage 3

In [6]:
meta=pd.read_csv(ROOT/'results/generated/metamorphic_summary.csv')
display(meta[meta.agent=='stage3'][['mutation','n','covered','correct','wrong','precision_if_covered']])

,mutation,n,covered,correct,wrong,precision_if_covered
0,background_note_pre,65,25,25,0,1.0
1,explicit_unrelated_post,65,25,25,0,1.0
2,explicit_unrelated_pre,65,25,25,0,1.0
3,explicit_year_post,65,25,25,0,1.0
4,fit_train_row_shuffle,35,35,35,0,1.0
5,independent_example_post,65,25,25,0,1.0
6,parenthetical_unrelated_pre,65,25,25,0,1.0
7,spacing_case,65,25,25,0,1.0
8,table_distractor_column,80,80,80,0,1.0


Les transformations ci-dessus sont conçues pour **ne pas changer la réponse mathématique** : phrase explicitement inutile, changement de casse/espaces, colonne Table inutile, permutation des lignes d'entraînement.

**MEASURED :** la partie déterministe de Stage 3 garde ici 0 faux positif. Ce résultat renforce la confiance dans les règles existantes ; il ne dit rien sur la qualité du LLM pour les tâches où Stage 3 s'abstient.

## 6. Hors distribution : ce que Stage 3 ne sait pas encore résoudre déterministiquement

In [7]:
ood=pd.read_csv(ROOT/'results/generated/ood_summary.csv')
display(ood[ood.agent=='stage3'])

,agent,dataset,n,covered,correct,wrong,coverage,precision_if_covered,selective_risk,deterministic_accuracy
0,stage3,ood1,80,0,0,0,0.000000,NaN,NaN,0.000000
1,stage3,ood2,120,20,20,0,0.166667,1.0,0.0,0.166667
2,stage3,ood3,80,8,0,8,0.100000,0.0,1.0,0.000000


- **OOD1** : Stage 3 couvre 0/80 des opérations ajoutées à notre benchmark de développement.
- **OOD2 holdout** : Stage 3 couvre 20/120, fait 20/20 sur ces 20 et s'abstient sur 100.
- **OOD3 fresh paraphrase** : Stage 3 intercepte 8/80 tâches mais les 8 sont fausses. Ce benchmark a été créé après v8 et révèle un risque important : une règle générique peut reconnaître le mauvais opérateur et produire un faux positif au lieu de s'abstenir.

L'abstention n'est donc pas un échec en soi : elle évite un faux positif déterministe. En revanche, elle signifie que le **fallback LLM doit être excellent** sur le résiduel.

## 7. Risques repérés à la lecture du code — hypothèses à tester, pas verdicts

In [8]:
source=(ROOT/'agents/champions/stage3_92_5.py').read_text(encoding='utf-8')
flags={
    'on_bad_lines="skip" présent':'on_bad_lines="skip"' in source or "on_bad_lines='skip'" in source,
    'conversion numérique seuil ~90%':'0.9' in source and 'str.match(_NUM_RE)' in source,
    'fake_read_csv présent':'fake_read_csv' in source,
    'to_float peut prendre le premier élément':'ravel()[0]' in source,
    'vote déterministe poids 100':'100.0' in source,
    'exec(...) historique présent':'exec(code, ns)' in source,
}
pd.DataFrame([{'risque / propriété':k,'présent dans le code':v} for k,v in flags.items()])

,risque / propriété,présent dans le code
0,"on_bad_lines=""skip"" présent",True
1,conversion numérique seuil ~90%,True
2,fake_read_csv présent,True
3,to_float peut prendre le premier élément,True
4,vote déterministe poids 100,True
5,exec(...) historique présent,True


Ces points sont des **pistes de red team** :

- `on_bad_lines="skip"` peut masquer une ligne mal parsée ;
- la conversion automatique d'une colonne peut confondre mesures et identifiants dans certains cas ;
- les fallbacks permissifs (`fake_read_csv`, coercion d'un non-scalaire vers son premier élément) peuvent transformer une erreur du LLM en réponse plausible mais fausse ;
- un résultat déterministe pondéré à 100 court-circuite pratiquement le vote, donc sa précision doit rester proche de 100% ;
- `exec(...)` est conservé dans le champion historique, mais la règle d'upload actuelle demande `dsh.run_python`.

**HYPOTHESIS :** ces mécanismes peuvent causer des erreurs privées. Nous ne les présentons pas comme des bugs prouvés tant qu'un test minimal ne les reproduit pas.

## 8. Conclusion de l’audit

Le champion 92,5 n’a pas besoin d’être réécrit : sa couche déterministe est **très précise sur le dev**. La suite doit donc être une démarche Champion/Challenger :

1. conserver Stage 3 comme référence privée ;
2. augmenter la couverture uniquement avec des règles à haute précision ;
3. essayer de casser chaque extension avec des transformations métamorphiques ;
4. distinguer benchmarks de développement et holdouts créés après les candidates ;
5. traiter les compositions inconnues par abstention/fallback plutôt que par interception agressive ;
6. mesurer runtime et contrat d’outils avant toute soumission.

Le notebook 2 reprend exactement à partir de cette conclusion et raconte les expériences **E00→E12**, jusqu’à la candidate locale **v10** et au blind check OOD5.